In [ ]:
# Instalar las librerías necesarias
!pip install pandas 
!pip install seaborn

In [ ]:

import seaborn as sns 
import matplotlib.pyplot as plt 

from sklearn.metrics import log_loss
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


In [ ]:

triaje = pd.read_csv(
    "/home/jovyan/datos/mimic-iv-ed-demo-2.2/mimic-iv-ed-demo-2.2/ed/triage.csv.gz"
)

df = triaje[triaje['acuity'].notna() & triaje['pain'].notna() &
            triaje['temperature'].notna() & triaje['heartrate'].notna() &
            triaje['o2sat'].notna()].drop(columns=['subject_id', 'stay_id']).copy()

# 0 = acuity 1-2, 1 = acuity 3-5 
y = (df['acuity'] < 3).astype(int).to_numpy()
print(np.unique(y, return_counts=True))

# diccionario para agrupar motivo de la consulta de la columna "chiefcomplaint"
# [sub] son palabras literales 
# [exact] son siglas que cuentan como una palabra completa
CONCEPTS = {
    'cc_arrest':   {'sub': ['arrest', 'intubat'], 'exact': []},
    'cc_neuro':    {'sub': ['droop', 'numb', 'weakness', 'dizz', 'diplopia', 'syncope', 'seizure',
                            'confusion', 'altered', 'letha', 'stroke', 'headache', 'hallucin',
                            'ambulate', 'tinnitus', 'head injury', 'head bleed'],
                    'exact': ['ams', 'cva', 'ich', 'sdh', 'sah']},
    'cc_cardiac':  {'sub': ['chest', 'palpitation', 'tachycardia', 'nstemi', 'aortic', 'atrial',
                            'ekg', 'hypertension', 'hypotension'], 'exact': []},
    'cc_resp':     {'sub': ['dyspnea', 'shortness', 'cough', 'hypox', 'laryngitis'],
                    'exact': ['sob', 'pe']},
    'cc_bleed':    {'sub': ['bleed', 'brbpr', 'hematemesis', 'hematuria', 'vomiting blood', 'emesis'],
                    'exact': []},
    'cc_metab':    {'sub': ['abnormal', 'lab', 'hyperglyc', 'hypoglyc', 'anemia', 'dehydrat',
                            'fatigue', 'hypotherm', 'neutropenia'],
                    'exact': ['dka', 'inr']},
    'cc_abd':      {'sub': ['abd', 'abdominal', 'n/v', 'nausea', 'vomit', 'diarrhea', 'epigastric',
                            'suprapubic', 'rlq', 'luq', 'ascites', 'rectal', 'urinary', 'dysuria',
                            'foley'], 'exact': []},
    'cc_infect':   {'sub': ['infect', 'fever', 'cellulitis', 'ulcer', 'abscess'],
                    'exact': ['ili']},
    'cc_trauma':   {'sub': ['fall', 'mvc', 'car vs', 'assault', 'injur', 'fracture', 'laceration',
                            'wound', 'trauma'], 'exact': ['fx']},
    'cc_psych':    {'sub': ['depress', 'anxiety', 'psych', 'etoh', 'overdose', 'insomnia'],
                    'exact': ['si']},
    'cc_vascular': {'sub': ['swelling'], 'exact': ['dvt']},
    'cc_localpain': {'sub': ['back', 'foot', 'knee', 'hand', 'shoulder', 'wrist', 'extremity'],
                     'exact': ['leg', 'arm', 'toe', 'rib', 'hip', 'ear', 'eye']},
    'cc_minor':    {'sub': ['eval', 'refill', 'picc', 'tube', 'hemodialysis', 'rash', 'allergic'],
                    'exact': []},
    'cc_transfer': {'sub': ['transfer'], 'exact': []},
    'cc_critical': {'sub': ['arrest', 'unresponsive', 'stroke', 'chest pain', 'shortness',
                            'seizure', 'bleeding'], 'exact': ['sob']},
}

# devuelve 1 si el texto contiene alguna palabra de [sub] o [exact] y 0 si no contiene
def has_concept(text, sub, exact):
    words = set(text.replace(',', ' ').replace('/', ' ').replace(';', ' ')
                    .replace('?', ' ').replace('-', ' ').replace('.', ' ').split())
    return int(any(k in text for k in sub) or any(w in words for w in exact))

# featuring 
VITALS = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain']

# convertir el dataframe en una tabla numérica para darsela al modelo
# limpiar de nulos
# rellenar huecos de variables con medias

def build_features(raw, medians=None, use_cc=True):
    d = raw.copy()

    for c in VITALS:
        d[c] = pd.to_numeric(d[c], errors='coerce')

    if medians is None:
        medians = d[VITALS].median()

    out = pd.DataFrame(index=d.index)
    out['pain_missing'] = d['pain'].isna().astype(int)   # antes de rellenar
    d['pain'] = d['pain'].fillna(0)
    d[VITALS] = d[VITALS].fillna(medians)

    for c in VITALS:
        out[c] = d[c]

    # Derivadas (temperatura en °F)
    shock = d['heartrate'] / d['sbp'].replace(0, np.nan)
    # feature derivada de recuencia cardíaca / presión sistólica
    out['feat_shock_index'] = shock.fillna(medians['heartrate'] / medians['sbp'])
    # feature derivada que suma sbp y resprate
    out['feat_qsofa_score'] = (d['sbp'] <= 100).astype(int) + (d['resprate'] >= 22).astype(int)

    out['flag_fever']       = (d['temperature'] >= 100.4).astype(int)
    out['flag_hypothermia'] = (d['temperature'] < 95.0).astype(int)
    out['flag_hypoxia']     = (d['o2sat'] < 92.0).astype(int)
    out['flag_tachycardia'] = (d['heartrate'] > 100.0).astype(int)
    out['flag_bradycardia'] = (d['heartrate'] < 60.0).astype(int)
    out['flag_tachypnea']   = (d['resprate'] >= 22.0).astype(int)
    out['flag_hypotension'] = (d['sbp'] < 90.0).astype(int)
    out['flag_severe_pain'] = (d['pain'] >= 7.0).astype(int)

    # Conceptos del chiefcomplaint
    if use_cc:
        txt = d['chiefcomplaint'].fillna('').str.lower()
        for col, spec in CONCEPTS.items():
            out[col] = txt.apply(lambda t: has_concept(t, spec['sub'], spec['exact']))
        out['cc_unknown'] = txt.apply(lambda t: int(t == '' or 'unknown' in t or t == 'sw'))
        out['feat_cc_n_symptoms'] = txt.apply(lambda t: t.count(',') + t.count(';') + 1)

    return out.astype('float32'), medians

# split y entrenamiento
df_train, df_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42, stratify=y)

Xtr, med = build_features(df_train)
Xte, _   = build_features(df_test, medians=med)

# modelo

# RESULTADO Test loss: 0.620  Test accuracy: 0.605
# rf = RandomForestClassifier(
#     n_estimators=300, 
#     max_depth=5,
#     min_samples_leaf=3, 
#     random_state=42,
#     max_features='sqrt',
#     max_samples=0.9,
#     oob_score=True
# )


# RESULTADO Test loss: 0.631  Test accuracy: 0.553
# rf = RandomForestClassifier(
#     n_estimators=300, 
#     max_depth=5,
#     min_samples_leaf=6, 
#     random_state=42,
#     max_features='sqrt',
#     max_samples=0.7,
#     oob_score=True
# )

# RESULTADO Test loss: 0.621  Test accuracy: 0.632
# rf = RandomForestClassifier(
#     n_estimators=300, 
#     max_depth=5,
#     min_samples_leaf=1, 
#     random_state=42,
#     max_features='sqrt',
#     max_samples=0.7,
#     oob_score=True
# )

# RESULTADO Test loss: 0.635  Test accuracy: 0.684
rf = RandomForestClassifier(
    n_estimators=300, 
    max_depth=5,
    min_samples_leaf=2, 
    random_state=42,
    max_features='sqrt',
    max_samples=0.2,
    oob_score=True
)

rf.fit(Xtr, y_train)
print(f"Columnas: {Xtr.shape[1]}  Accuracy: {rf.score(Xte, y_test):.3f}")

# importancia relativa de cada variable dentro del modelo una vez entrenado
imp = pd.Series(rf.feature_importances_, index=Xtr.columns).sort_values(ascending=False)
print(imp.head(15))

test_accuracy = rf.score(Xte, y_test)
test_loss = log_loss(y_test, rf.predict_proba(Xte))
print(f"Test loss: {test_loss:.3f}  Test accuracy: {test_accuracy:.3f}")


# Evaluar diferentes hiperparámetros en el modelo
# from itertools import product

# grid = {
#     'max_depth':        [3, 5, 8],
#     'min_samples_leaf': [1, 3, 6],
#     'max_features':     ['sqrt', 0.5],
#     'max_samples':      [0.7, 0.9],
# }

# resultados = []
# for md, msl, mf, ms in product(*grid.values()):
#     m = RandomForestClassifier(
#         n_estimators=300, max_depth=md, min_samples_leaf=msl,
#         max_features=mf, max_samples=ms,
#         oob_score=True, random_state=42, n_jobs=-1
#     )
#     m.fit(Xtr, y_train)
#     resultados.append((m.oob_score_, md, msl, mf, ms))

# res = pd.DataFrame(resultados, columns=['oob', 'max_depth', 'min_samples_leaf',
#                                         'max_features', 'max_samples'])
# print(res.sort_values('oob', ascending=False).head(10))

